# Fase 1: Módulo de Filtros Fundamentais e Solvência Financeira

Este notebook demonstra o fluxo de trabalho quantitativo para o cálculo de rácios de liquidez, estrutura de capital, rentabilidade e aplicação do **Filtro Passa-Baixo Fundamental (Defensive Graham Screen)**.

### Especificação Matemática dos Indicadores
- **Current Ratio**: $\text{Current Ratio} = \frac{\text{Ativo Corrente}}{\text{Passivo Corrente}}$ (Filtro base $\ge 1,5$ ou $\ge 2,0$)
- **Debt-to-Equity**: $\text{Debt-to-Equity} = \frac{\text{Passivo Total}}{\text{Capital Próprio}}$
- **Earnings Yield**: $\text{Earnings Yield} = \frac{\text{LPA}}{\text{Preço de Fecho}}$
- **Price-to-Book ($P/B$)**: $P/B = \frac{\text{Capitalização de Mercado}}{\text{Valor Contabilístico}}$
- **Return on Assets ($ROA$)**: $ROA = \frac{\text{Resultado Líquido}}{\text{Ativo Total}}$
- **Free Cash Flow Yield**: $\text{FCF Yield} = \frac{\text{Fluxo de Caixa Livre}}{\text{Enterprise Value}}$
- **Net Income Growth (5Y)**: $\text{CAGR}_{5Y} = \left(\frac{\text{Net Income}_t}{\text{Net Income}_{t-5}}\right)^{1/5} - 1$

In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Adicionar diretório raiz ao path
sys.path.insert(0, os.path.abspath('..'))

from src.features.fundamentals import FundamentalScreener

In [2]:
# Simulação de um universo de 5 ativos
data = {
    'ticker': ['AAPL', 'MSFT', 'GOOGL', 'WEAK_CO', 'RISKY_CO'],
    'current_assets': [130000, 180000, 150000, 40000, 10000],
    'current_liabilities': [60000, 70000, 60000, 50000, 15000],  # WEAK_CO Current Ratio < 1.0
    'total_liabilities': [100000, 120000, 80000, 90000, 80000],
    'total_equity': [80000, 100000, 120000, 20000, 10000],       # RISKY_CO Debt/Equity = 8.0
    'eps': [6.0, 9.5, 5.8, -0.5, 0.2],
    'price': [175.0, 330.0, 135.0, 12.0, 5.0],
    'market_cap': [2800000, 2400000, 1700000, 100000, 20000],
    'book_value': [80000, 100000, 120000, 20000, 10000],
    'net_income': [100000, 72000, 60000, -2000, 500],
    'total_assets': [350000, 380000, 300000, 80000, 30000],
    'free_cash_flow': [90000, 65000, 55000, -1000, 200],
    'enterprise_value': [2850000, 2450000, 1650000, 120000, 30000],
    'net_income_5y_ago': [60000, 40000, 35000, 5000, -1000]
}

df_universe = pd.DataFrame(data)
df_universe

In [3]:
# Inicialização e Cálculo dos Indicadores
screener = FundamentalScreener(df_universe)
df_computed = screener.compute_all_metrics()
df_filtered, mask = screener.apply_solvency_filter(min_current_ratio=1.5, max_debt_equity=2.0, min_roa=0.0)
scores = screener.calculate_graham_quality_score()

# Exibição dos Ativos Aprovados
print("=== UNIVERSO DE ATIVOS FILTRADOS (SOLVENTES E QUALIFICADOS) ===")
print(df_filtered[['ticker', 'current_ratio', 'debt_to_equity', 'earnings_yield', 'roa']])

In [4]:
# Relatório Consolidado de Validação
report_df = pd.DataFrame({
    'Ticker': df_computed['ticker'],
    'Current Ratio (≥1.5)': df_computed['current_ratio'].round(2),
    'Debt-to-Equity (≤2.0)': df_computed['debt_to_equity'].round(2),
    'ROA (>0%)': (df_computed['roa'] * 100).round(2).astype(str) + '%',
    'Status do Filtro': np.where(mask, 'Aprovado', 'Rejeitado'),
    'Score Fundamental (0-100)': df_computed['fundamental_quality_score'].round(1)
})

report_df